# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. 

The dataset focuses on clinical and pathological characteristics of 77 cancer survivors with second primary colorectal cancer, including MSI-H status and anatomical distributions.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset package
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset metadata
print('Dataset name:', metadata.name)
print('Description:', metadata.description)
print('Identifier:', getattr(metadata, 'identifier', '(none)'))
print('Date published:', getattr(metadata, 'datePublished', '(none)'))
print('Version:', getattr(metadata, 'version', '(none)'))

## 2. Data Overview
Review available record sets, their fields, and column IDs.

All entities are referenced by their `@id` fields.

In [ ]:
# List all record sets and fields using IDs
from mlcroissant._src.structure.types import RecordSet

record_sets = list(dataset.record_sets.values())

if not record_sets:
    print('No record sets defined in the Croissant schema.')
else:
    for rs in record_sets:
        print(f"\nRecord set: {rs.metadata['@id']} | Name: {rs.metadata.get('name', '(Unnamed)')}")
        # List fields/columns
        if hasattr(rs, 'fields') and rs.fields:
            print('  Fields:')
            for field_obj in rs.fields:
                print(f"    {field_obj['@id']} | Label: {field_obj.get('name', '(No name)')} | Data Type: {field_obj.get('dataType', '(N/A)')}")
        elif hasattr(rs, 'columns') and rs.columns:
            print('  Columns:')
            for col_obj in rs.columns:
                print(f"    {col_obj['@id']} | Label: {col_obj.get('name', '(No name)')} | Data Type: {col_obj.get('dataType', '(N/A)')}")
        else:
            print("  No fields or columns found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record set and field references use their `@id`.

In [ ]:
# We'll get all available record set @id values:
rs_ids = [rs.metadata['@id'] for rs in dataset.record_sets.values()]
print('Record set IDs:', rs_ids)

# For this dataset, there is usually one main tabular record set. We will load ALL dataframes.
dataframes = {}
for rs_id in rs_ids:
    print(f"Loading records for: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records. First columns: {df.columns[:5].tolist()}")
        else:
            print("No records found in this record set.")
    except Exception as e:
        print(f"Error loading data for {rs_id}: {e}")

# Show columns of the first DataFrame (assuming main data is in the first record set):
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print('\nAvailable columns in main record set:', list(dataframes[main_rs_id].columns))
    display(dataframes[main_rs_id].head())
else:
    print('No dataframes loaded.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping by a key attribute, all by referencing fields by their `@id`.

In [ ]:
# Select main record set and choose a numeric field and group field by `@id`
if not dataframes:
    print('No dataframes available for EDA.')
else:
    df = dataframes[main_rs_id]
    print('Columns:')
    pprint.pprint(list(df.columns))
    
    # We'll attempt to find a suitable numeric field (e.g. 'age' or diagnosis interval in years)
    # You may need to update these with the exact @id from the data overview cell.
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        # Heuristics: choose 'age', 'interval', 'years', or similar
        if 'age' in col.lower() or 'interval' in col.lower() or 'year' in col.lower():
            numeric_field_id = col
            break
    # Heuristic for group: use 'sex', 'msi', or 'anatomical' for grouping.
    for col in df.columns:
        if 'sex' in col.lower() or 'msi' in col.lower() or 'location' in col.lower():
            group_field_id = col
            break
    print('Chosen numeric field:', numeric_field_id)
    print('Chosen group field:', group_field_id)
    
    # In case field is not found, skip steps
    if numeric_field_id is not None and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        threshold = df[numeric_field_id].quantile(0.75)
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered {len(filtered_df)} records where {numeric_field_id} > {threshold:.2f}")
        
        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouping
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name=f"mean_{numeric_field_id}")
            print(grouped_df.head())
    else:
        print("No suitable numeric field found for EDA, or selected field is not numeric.")

## 5. Visualization
Visualize the distribution of a numeric variable and relationships by group.
All field references use their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id is not None and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    # Histogram
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

    # Boxplot by group
    if group_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.tight_layout()
        plt.show()
else:
    print("No suitable numeric field or group found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load a Croissant dataset and access its records using `mlcroissant`. All dataset entities were referenced by `@id`. An overview of the dataset and columns was provided, records were loaded into DataFrames, and basic filtering, normalization, and grouping (by `@id`) were performed.

These steps can be adapted for further statistical modeling, advanced analysis, and reporting using the dataset.

**Note:** For production use, always consult the data dictionary to map `@id` fields to user-friendly names, and validate field types before processing.